In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset, Subset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import LassoCV
from sklearn.model_selection import TimeSeriesSplit

# --- ARCHITECTURE ---

class NHITSBlock(nn.Module):
    def __init__(self, input_dim, seq_len, max_horizon, pool_size, n_theta, hidden_dim, dropout_rate):
        super().__init__()
        self.pool_size = pool_size
        self.seq_len = seq_len
        self.input_dim = input_dim
        self.max_horizon = max_horizon
        self.pooled_seq_len = int(np.ceil(seq_len / pool_size))
        flat_dim = input_dim * self.pooled_seq_len

        self.mlp = nn.Sequential(
            nn.Linear(flat_dim, hidden_dim),
            nn.LeakyReLU(0.1),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.1)
        )

        self.theta_b = nn.Linear(hidden_dim, seq_len * input_dim)
        self.theta_f = nn.Linear(hidden_dim, n_theta)

    def forward(self, x):
        x_pool = x.permute(0, 2, 1) 
        if self.pool_size > 1:
            x_pool = F.max_pool1d(x_pool, kernel_size=self.pool_size, stride=self.pool_size, ceil_mode=True)
        h = self.mlp(x_pool.reshape(x_pool.size(0), -1))
        backcast = self.theta_b(h).reshape(-1, self.seq_len, self.input_dim)
        theta_f = self.theta_f(h).unsqueeze(1) 
        forecast = F.interpolate(theta_f, size=self.max_horizon, mode='linear', align_corners=True).squeeze(1)
        return backcast, forecast

class RealNHITS(nn.Module):
    def __init__(self, input_dim, seq_len, hidden_dim=256, dropout_rate=0.1):
        super().__init__()
        self.max_horizon = 24 
        self.block1 = NHITSBlock(input_dim, seq_len, self.max_horizon, 4, 6, hidden_dim, dropout_rate)
        self.block2 = NHITSBlock(input_dim, seq_len, self.max_horizon, 2, 12, hidden_dim, dropout_rate)
        self.block3 = NHITSBlock(input_dim, seq_len, self.max_horizon, 1, 24, hidden_dim, dropout_rate)

    def forward(self, x):
        exog = x[:, :, 1:] 
        target = x[:, :, 0:1] 
        b1, f1 = self.block1(x)
        res_target = target - b1[:, :, 0:1] 
        input_2 = torch.cat([res_target, exog], dim=-1)
        b2, f2 = self.block2(input_2)
        res_target = res_target - b2[:, :, 0:1]
        input_3 = torch.cat([res_target, exog], dim=-1)
        _, f3 = self.block3(input_3)
        return (f1 + f2 + f3)[:, 3]

class AQDataset(Dataset):
    def __init__(self, data, seq_len, horizon=4):
        self.data = data
        self.seq_len = seq_len
        self.horizon = horizon
    def __len__(self):
        return len(self.data) - self.seq_len - self.horizon
    def __getitem__(self, idx):
        x = torch.FloatTensor(self.data[idx : idx + self.seq_len])
        y = torch.FloatTensor([self.data[idx + self.seq_len + self.horizon - 1, 0]])
        return x, y

def get_metrics(a, p):
    return np.sqrt(mean_squared_error(a, p)), mean_absolute_error(a, p)

# --- EXECUTION ---

if __name__ == "__main__":
    device = torch.device("cpu") 
    data_path = "data/data_clean/cleaned_data_toronto_downtown.csv" 
    df = pd.read_csv(data_path, low_memory=False)
    features = ['PM_ppb', 'Temp (°C)', 'Rel Hum (%)', 'Wind Spd (km/h)', 'Stn Press (kPa)', 
                'Dew Point Temp (°C)', 'Precip. Amount (mm)', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos']
    
    train_size = int(len(df) * 0.7)
    scaler = StandardScaler()
    scaler.fit(df.iloc[:train_size][features])
    scaled_data_array = scaler.transform(df[features])
    pm_mean, pm_std = scaler.mean_[0], scaler.scale_[0]

    BEST_LOOKBACK, BEST_HIDDEN, BEST_DROP = 24, 512, 0.1
    
    dataset = AQDataset(scaled_data_array, seq_len=BEST_LOOKBACK)
    n = len(dataset)
    train_idx, val_idx = int(0.7 * n), int(0.85 * n)
    
    val_loader = DataLoader(Subset(dataset, range(train_idx, val_idx)), batch_size=128)
    test_loader = DataLoader(Subset(dataset, range(val_idx, n)), batch_size=128)

    # 1. LASSO Baseline
    print("Training LASSO Baseline...")
    X_all = np.array([scaled_data_array[i : i + BEST_LOOKBACK].flatten() for i in range(n)])
    y_all = np.array([scaled_data_array[i + BEST_LOOKBACK + 3, 0] for i in range(n)])
    X_train_lasso, y_train_lasso = X_all[:val_idx], y_all[:val_idx]
    X_test_lasso = X_all[val_idx:]
    
    lasso = LassoCV(cv=TimeSeriesSplit(n_splits=5), random_state=42, max_iter=10000, n_jobs=-1)
    lasso.fit(X_train_lasso, y_train_lasso)
    lasso_preds = (lasso.predict(X_test_lasso) * pm_std) + pm_mean

    # 2. N-HiTS Inference
    print("Running N-HiTS Inference...")
    model = RealNHITS(input_dim=11, seq_len=BEST_LOOKBACK, hidden_dim=BEST_HIDDEN, dropout_rate=BEST_DROP)
    
    # Calculate parameters
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Trainable Parameters: {total_params:,}")
    
    model.load_state_dict(torch.load("models/global_champion_nhits.pt", map_location=device))
    model.eval()

    val_preds, val_acts = [], []
    with torch.no_grad():
        for x, y in val_loader:
            val_preds.append(model(x).numpy().flatten())
            val_acts.append(y.numpy().flatten())
    
    val_residuals = np.concatenate(val_acts) - np.concatenate(val_preds)
    margin_of_error = 1.96 * np.std(val_residuals) * pm_std

    test_preds, test_acts = [], []
    with torch.no_grad():
        for x, y in test_loader:
            test_preds.append(model(x).numpy().flatten())
            test_acts.append(y.numpy().flatten())
            
    unscaled_preds = (np.concatenate(test_preds) * pm_std) + pm_mean
    unscaled_actuals = (np.concatenate(test_acts) * pm_std) + pm_mean

    # Results
    l_rmse, l_mae = get_metrics(unscaled_actuals, lasso_preds)
    n_rmse, n_mae = get_metrics(unscaled_actuals, unscaled_preds)

    print(f"\n--- RESULTS (N={len(unscaled_actuals)}) ---")
    print(f"LASSO       | RMSE: {l_rmse:.4f} | MAE: {l_mae:.4f}")
    print(f"N-HiTS      | RMSE: {n_rmse:.4f} | MAE: {n_mae:.4f}")

    # Plotting
    PLOT_LEN = 300
    plt.figure(figsize=(15, 8))
    
    plt.fill_between(range(PLOT_LEN), 
                     unscaled_preds[:PLOT_LEN] - margin_of_error, 
                     unscaled_preds[:PLOT_LEN] + margin_of_error, 
                     color='red', alpha=0.1, label="N-HiTS 95% Confidence Interval")
    
    plt.plot(unscaled_actuals[:PLOT_LEN], label="Actual", color='black', alpha=0.6)
    plt.plot(lasso_preds[:PLOT_LEN], label="LASSO", color='green', linestyle='-.')
    plt.plot(unscaled_preds[:PLOT_LEN], label="N-HiTS", color='red', linewidth=2)
    
    plt.title(f"Toronto PM2.5 300 Hour Forecast Comparison\n(Model Parameters: {total_params:,})")
    plt.legend()
    plt.savefig("results/final_comparison_no_persistence.png", dpi=300)
    print("Plot saved to results/final_comparison_no_persistence.png")

Training LASSO Baseline...
Running N-HiTS Inference...
Total Trainable Parameters: 1,453,890


FileNotFoundError: [Errno 2] No such file or directory: 'global_champion_nhits.pt'

In [3]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

total_params = count_parameters(model)
# update
print(f"Total Trainable Parameters: {total_params:,}")

Total Trainable Parameters: 1,453,890
